# 03 — Training: benchmark + LogReg / RF / XGBoost in SageMaker

**Team Update 4.1 steps 1–2.** Runs the simple benchmark, then trains the
three design-doc candidates as SageMaker Training jobs on the temporally-split
feature table. The champion selection happens in notebook 04.

| Step | Verify (📸) |
|---|---|
| Split materialization (Task 8) | 4 split CSVs in S3 |
| Benchmark on the full table | benchmark metrics printed |
| Training jobs | completed-job table + metrics comparison |

**Run order:** top to bottom in SageMaker Studio. Training uses
`ml.m5.2xlarge` (XGBoost) and `ml.m5.xlarge` (sklearn) per the design doc;
three seeded XGBoost runs over the design-doc grid.

## 1. Setup

In [1]:
%pip install -q "sagemaker<3" xgboost

import json
import os
import tarfile
from pathlib import Path

import boto3
import pandas as pd
import sagemaker
from sagemaker import image_uris
from sagemaker.inputs import TrainingInput
from sagemaker.session import Session
from sagemaker.sklearn import SKLearn
from sagemaker.estimator import Estimator

region = boto3.Session().region_name
boto_session = boto3.Session(region_name=region)
sm_client = boto_session.client("sagemaker", region_name=region)
sess = Session(boto_session=boto_session, sagemaker_client=sm_client)
try:
    role = sagemaker.get_execution_role()  # Studio / notebook instance
except Exception:
    # local run (personal account): dedicated project role if present,
    # otherwise the newest SageMaker execution role in the account
    iam = boto3.client("iam", region_name=region)
    by_name = {r["RoleName"]: r for r in iam.list_roles()["Roles"]}
    if "aai-540-g1-execution-role" in by_name:
        picked = "aai-540-g1-execution-role"
    else:
        cands = [n for n in by_name if n.startswith("AmazonSageMaker-ExecutionRole")]
        if not cands:
            raise RuntimeError("no SageMaker execution role found - run scripts/ensure_execution_role.py")
        picked = sorted(cands, key=lambda n: by_name[n]["CreateDate"])[-1]
    account = boto3.client("sts").get_caller_identity()["Account"]
    role = f"arn:aws:iam::{account}:role/{picked}"
print(f"role={role}")
bucket = sess.default_bucket()
prefix = "aai-540-g1"
s3 = boto3.client("s3", region_name=region)

print(f"role={role}\nbucket={bucket}")

Note: you may need to restart the kernel to use updated packages.


sagemaker.config INFO - Not applying SDK defaults from location: /Library/Application Support/sagemaker/config.yaml


sagemaker.config INFO - Not applying SDK defaults from location: /Users/sourangshupal/Library/Application Support/sagemaker/config.yaml


/Users/sourangshupal/Downloads/final-project-usd/.venv-sm/lib/python3.9/site-packages/sagemaker/__init__.py:86: SageMakerV2DeprecationWarning: You are using the SageMaker Python SDK v2, which is on path of deprecation. v3 is the actively developed major version.
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.
  warn_v2_deprecation()
You are using the SageMaker Python SDK v2, which is on path of deprecation. v3 is the actively developed major version.
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.


Couldn't call 'get_role' to get Role ARN from role name final-usd to get Role path.


/Users/sourangshupal/Downloads/final-project-usd/.venv-sm/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


role=arn:aws:iam::685057748560:role/aai-540-g1-execution-role


role=arn:aws:iam::685057748560:role/aai-540-g1-execution-role
bucket=sagemaker-us-east-1-685057748560


## 2. Ensure the repo code is present in Studio

The notebook drives `src/train.py` / `src/benchmark.py` from the repo. Clone
it if this Studio file view doesn't already have it.

In [2]:
import os
from pathlib import Path

# CWD depends on the runner: Studio file view starts at the repo root, while
# `jupyter nbconvert` starts in the notebook's directory. Resolve the repo
# root (the dir that contains src/) and work from there.
if (Path.cwd() / "src").is_dir():
    REPO = Path.cwd()
elif (Path.cwd().parent / "src").is_dir():
    REPO = Path.cwd().parent
else:  # fresh Studio file view: clone the repo next to the notebooks dir
    !git clone --quiet https://github.com/University-San-Diego-MAAI/AAI-540-Group-1-Final-Project.git ../repo
    REPO = Path.cwd().parent / "repo"
os.chdir(REPO)
print("repo root:", REPO, "| src/train.py:", (REPO / "src" / "train.py").is_file())

repo root: /Users/sourangshupal/Downloads/final-project-usd | src/train.py: True


## 3. Task 8 — materialize the four split CSVs to S3

Feature-store-style dataset extraction: the training table carries the course
`split` column; we persist each slice as its own CSV so downstream jobs and
monitors can reference a stable per-split URI.

In [3]:
s3.download_file(bucket, f"{prefix}/processed/training/training_table.csv", "training_table.csv")
df = pd.read_csv("training_table.csv")

for split_name in ["train", "validation", "test", "production"]:
    part = df[df["split"] == split_name]
    local = f"split_{split_name}.csv"
    part.to_csv(local, index=False)
    s3.upload_file(local, bucket, f"{prefix}/processed/splits/{local}")
    print(f"{split_name}: {len(part):,} rows -> s3://{bucket}/{prefix}/processed/splits/{local}")

train: 33,050 rows -> s3://sagemaker-us-east-1-685057748560/aai-540-g1/processed/splits/split_train.csv


validation: 8,263 rows -> s3://sagemaker-us-east-1-685057748560/aai-540-g1/processed/splits/split_validation.csv


test: 8,263 rows -> s3://sagemaker-us-east-1-685057748560/aai-540-g1/processed/splits/split_test.csv


production: 33,048 rows -> s3://sagemaker-us-east-1-685057748560/aai-540-g1/processed/splits/split_production.csv


## 4. Team Update 4.1 step 1 — benchmark on the full table

Simple heuristics only (majority class + history-visits tercile rule). The
real models must beat these on the test split.

In [4]:
!python src/benchmark.py --table training_table.csv --output-dir artifacts/evaluation
s3.upload_file("artifacts/evaluation/benchmark_metrics.json", bucket,
               f"{prefix}/artifacts/evaluation/benchmark_metrics.json")

wrote artifacts/evaluation/benchmark_metrics.json
{
  "b1_majority_class": {
    "predicted_class": "Low",
    "macro_f1": 0.1681,
    "accuracy": 0.337,
    "per_class": {
      "Low": {
        "precision": 0.337,
        "recall": 1.0,
        "f1": 0.5042
      },
      "Medium": {
        "precision": 0.0,
        "recall": 0.0,
        "f1": 0.0
      },
      "High": {
        "precision": 0.0,
        "recall": 0.0,
        "f1": 0.0
      }
    },
    "high_risk_recall": 0.0,
    "confusion_matrix": [
      [
        2785,
        0,
        0
      ],
      [
        2750,
        0,
        0
      ],
      [
        2728,
        0,
        0
      ]
    ],
    "roc_auc_ovr": 0.5
  },
  "b2_visits_tercile_rule": {
    "macro_f1": 0.5304,
    "accuracy": 0.5329,
    "per_class": {
      "Low": {
        "precision": 0.6113,
        "recall": 0.6115,
        "f1": 0.6114
      },
      "Medium": {
        "precision": 0.3663,
        "recall": 0.3465,
        "f1": 0.3561
   

## 5. Training runs

`src/train.py` reads the full table from the `train` channel, derives the
splits internally, early-stops XGBoost on the validation slice, and writes
`<model>_metrics.json` into the model artifact. Hyperparameter keys use
dashes because SageMaker passes them to argparse verbatim.

In [5]:
import time

TRAIN_CHANNEL = TrainingInput(
    f"s3://{bucket}/{prefix}/processed/training/training_table.csv",
    content_type="text/csv",
)
XGB_IMAGE = image_uris.retrieve("xgboost", region, version="1.7-1")

RUN_SUFFIX = str(int(time.time()))

XGB_GRID = {
    "xgb-e01-d6-w5":  {"eta": "0.1",  "max-depth": "6", "min-child-weight": "5"},
    "xgb-e005-d6-w5": {"eta": "0.05", "max-depth": "6", "min-child-weight": "5"},
    "xgb-e01-d8-w1":  {"eta": "0.1",  "max-depth": "8", "min-child-weight": "1"},
}


def run_training_job(name, estimator_cls, hp, instance, image=None):
    kwargs = dict(
        entry_point="train.py",  # relative to source_dir="src"
        source_dir="src",
        role=role,
        instance_count=1,
        instance_type=instance,
        output_path=f"s3://{bucket}/{prefix}/artifacts/training/{name}",
        sagemaker_session=sess,
        hyperparameters={"model": hp.pop("model", "xgb"), **hp},
    )
    if estimator_cls is SKLearn:
        est = SKLearn(framework_version="0.23-1", **kwargs)
    else:
        est = Estimator(image_uri=image, **kwargs)
    job_name = f"{name}-{RUN_SUFFIX}"  # names are one-use-only in SageMaker
    est.fit({"train": TRAIN_CHANNEL}, job_name=job_name, wait=True)
    return f"s3://{bucket}/{prefix}/artifacts/training/{name}/{job_name}/output/model.tar.gz"


sklearn_runs = {
    "logreg-c1":  {"model": "logreg", "c-reg": "1.0"},
    "rf-d16-l5":  {"model": "rf", "n-estimators": "300", "max-depth": "16",
                   "min-samples-leaf": "5"},
}

### 5a. sklearn baselines (script mode, ml.m5.xlarge)

In [6]:
model_artifacts = {}
for name, hp in sklearn_runs.items():
    print(f"=== {name} ===")
    model_artifacts[name] = run_training_job(name, SKLearn, dict(hp), "ml.m5.xlarge")

/Users/sourangshupal/Downloads/final-project-usd/.venv-sm/lib/python3.9/site-packages/sagemaker/estimator.py:588: SageMakerV2DeprecationWarning: SKLearn is part of the SageMaker Python SDK v2, which is on path of deprecation. In v3, use `ModelTrainer` (`from sagemaker.train import ModelTrainer`).
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.
  warn_v2_deprecation(
SKLearn is part of the SageMaker Python SDK v2, which is on path of deprecation. In v3, use `ModelTrainer` (`from sagemaker.train import ModelTrainer`).
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.


=== logreg-c1 ===


/Users/sourangshupal/Downloads/final-project-usd/.venv-sm/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


INFO:sagemaker:Creating training-job with name: logreg-c1-1791300001


2026-10-06 15:20:06 Starting - Starting the training job

...


2026-10-06 15:24:48 Training - Training image download completed. Training in progress.

.

2026-10-06 15:25:00,328 sagemaker-containers INFO     Imported framework sagemaker_sklearn_container.training
2026-10-06 15:25:00,332 sagemaker-training-toolkit INFO     No GPUs detected (normal if no gpus installed)
2026-10-06 15:25:00,375 sagemaker_sklearn_container.training INFO     Invoking user training script.
2026-10-06 15:25:00,591 sagemaker-training-toolkit INFO     No GPUs detected (normal if no gpus installed)
2026-10-06 15:25:00,604 sagemaker-training-toolkit INFO     No GPUs detected (normal if no gpus installed)
2026-10-06 15:25:00,617 sagemaker-training-toolkit INFO     No GPUs detected (normal if no gpus installed)
2026-10-06 15:25:00,625 sagemaker-training-toolkit INFO     Invoking user script
Training Env:
{
    "additional_framework_parameters": {},
    "channel_input_dirs": {
        "train": "/opt/ml/input/data/train"
    },
    "current_host": "algo-1",
    "framework_module": "sagemaker_sklearn_container.training:main",
    "hosts": [
        "algo-1"
    ],
    


2026-10-06 15:25:26 Uploading - Uploading generated training model
2026-10-06 15:25:26 Completed - Training job completed


Training seconds: 83
Billable seconds: 83


### 5b. XGBoost grid (ml.m5.2xlarge, primary model)

In [7]:
for name, hp in XGB_GRID.items():
    print(f"=== {name} ===")
    model_artifacts[name] = run_training_job(name, Estimator, dict(hp),
                                             "ml.m5.2xlarge", image=XGB_IMAGE)
print(json.dumps(model_artifacts, indent=2))

/Users/sourangshupal/Downloads/final-project-usd/.venv-sm/lib/python3.9/site-packages/sagemaker/estimator.py:588: SageMakerV2DeprecationWarning: Estimator is part of the SageMaker Python SDK v2, which is on path of deprecation. In v3, use `ModelTrainer` (`from sagemaker.train import ModelTrainer`).
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.
  warn_v2_deprecation(
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.


INFO:sagemaker.telemetry.telemetry_logging:SageMaker Python SDK will collect telemetry to help us better understand our user's needs, diagnose issues, and deliver additional features.
To opt out of telemetry, please disable via TelemetryOptOut parameter in SDK defaults config. For more information, refer to https://sagemaker.readthedocs.io/en/stable/overview.html#configuring-and-using-defaults-with-the-sagemaker-python-sdk.


=== xgb-e01-d6-w5 ===


INFO:sagemaker:Creating training-job with name: xgb-e01-d6-w5-1791300001


2026-10-06 15:26:09 Starting - Starting the training job

...


2026-10-06 15:34:50 Uploading - Uploading generated training model

/miniconda3/lib/python3.9/site-packages/sagemaker_containers/_server.py:22: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
[2026-10-06 15:34:40.875 ip-10-0-235-74.ec2.internal:7 INFO utils.py:28] RULE_JOB_STOP_SIGNAL_FILENAME: None
[2026-10-06 15:34:40.947 ip-10-0-235-74.ec2.internal:7 INFO profiler_config_parser.py:111] User has disabled profiler.
[2026-10-06:15:34:41:INFO] Imported framework sagemaker_xgboost_container.training
[2026-10-06:15:34:41:INFO] No GPUs detected (normal if no gpus installed)
[2026-10-06:15:34:41:INFO] Invoking user training script.
[2026-10-06:15:34:41:INFO] Module train does not provide a setup.py. 
Generating setup.py
[2026-10-06:15:34:41:INFO] Generating setup.cfg
[2026-10-06:15:34:41:INFO] Generating MANIFEST.in
[2026-10-06:15:34:41:INFO] I


2026-10-06 15:35:03 Completed - Training job completed


Training seconds: 99
Billable seconds: 99


{
  "logreg-c1": "s3://sagemaker-us-east-1-685057748560/aai-540-g1/artifacts/training/logreg-c1/logreg-c1-1791300001/output/model.tar.gz",
  "rf-d16-l5": "s3://sagemaker-us-east-1-685057748560/aai-540-g1/artifacts/training/rf-d16-l5/rf-d16-l5-1791300001/output/model.tar.gz",
  "xgb-e01-d6-w5": "s3://sagemaker-us-east-1-685057748560/aai-540-g1/artifacts/training/xgb-e01-d6-w5/xgb-e01-d6-w5-1791300001/output/model.tar.gz",
  "xgb-e005-d6-w5": "s3://sagemaker-us-east-1-685057748560/aai-540-g1/artifacts/training/xgb-e005-d6-w5/xgb-e005-d6-w5-1791300001/output/model.tar.gz",
  "xgb-e01-d8-w1": "s3://sagemaker-us-east-1-685057748560/aai-540-g1/artifacts/training/xgb-e01-d8-w1/xgb-e01-d8-w1-1791300001/output/model.tar.gz"
}


## 6. 📊 Metrics comparison (screenshot for the tracker)

Pulls every `<model>_metrics.json` out of the model tarballs and ranks the
runs by validation macro F1 (champion rule: best validation macro F1 subject
to a High-Risk recall floor; notebook 04 confirms on the test split).

In [8]:
import shutil


def fetch_metrics(artifact_s3):
    local_tar = Path("model.tar.gz")
    key = artifact_s3.split(f"s3://{bucket}/")[1]
    s3.download_file(bucket, key, str(local_tar))
    with tarfile.open(local_tar) as tar:
        member = next(m for m in tar.getnames() if m.endswith("_metrics.json"))
        metrics = json.loads(tar.extractfile(member).read())
    local_tar.unlink()
    return metrics


rows = []
for name, artifact_s3 in model_artifacts.items():
    m = fetch_metrics(artifact_s3)
    rows.append({
        "run": name,
        "model": m["model"],
        "val_macro_f1": m["validation"]["macro_f1"],
        "val_high_recall": m["validation"]["high_risk_recall"],
        "test_macro_f1": m["test"]["macro_f1"],
        "test_high_recall": m["test"]["high_risk_recall"],
        "test_roc_auc": m["test"]["roc_auc_ovr"],
    })
comparison = (pd.DataFrame(rows)
              .sort_values("val_macro_f1", ascending=False)
              .reset_index(drop=True))
comparison

# persist for notebook 04
json.dump(model_artifacts, open("artifacts/evaluation/model_artifacts.json", "w"), indent=2)
s3.upload_file("artifacts/evaluation/model_artifacts.json", bucket,
               f"{prefix}/artifacts/evaluation/model_artifacts.json")
comparison.to_csv("artifacts/evaluation/training_comparison.csv", index=False)
s3.upload_file("artifacts/evaluation/training_comparison.csv", bucket,
               f"{prefix}/artifacts/evaluation/training_comparison.csv")